In [1]:
import subprocess

# Colab bootstrap (idempotent)
subprocess.run('apt-get update -qq > /dev/null', shell=True, check=True)
subprocess.run('apt-get install -y openjdk-21-jdk-headless -qq > /dev/null', shell=True, check=True)
subprocess.run('python -m pip install -q pyspark==4.0.1 findspark plotly openpyxl kaggle', shell=True, check=True)

CompletedProcess(args='python -m pip install -q pyspark==4.0.1 findspark plotly openpyxl kaggle', returncode=0)

In [2]:
import glob
import os

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'

spark_candidates = sorted(glob.glob('/content/spark-*'))
if spark_candidates:
    os.environ['SPARK_HOME'] = spark_candidates[-1]

print('JAVA_HOME =', os.environ.get('JAVA_HOME'))
print('SPARK_HOME =', os.environ.get('SPARK_HOME', '<pyspark-managed>'))

JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64
SPARK_HOME = <pyspark-managed>


In [3]:
import findspark
findspark.init()

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('MinecraftComplexity')
    .master('local[*]')
    .config('spark.sql.repl.eagerEval.enabled', 'true')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.adaptive.skewJoin.enabled', 'true')
    .config('spark.sql.autoBroadcastJoinThreshold', str(32 * 1024 * 1024))
    .config('spark.sql.shuffle.partitions', '64')
    .config('spark.default.parallelism', '64')
    .config('spark.driver.memory', '8g')
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir('/content/spark_checkpoints')
print('Spark version:', spark.version)

Spark version: 4.0.1


In [5]:
spark

## Spark ui, ngrok tunnel and kaggle token

I remove all of them because we don't need them, it's easier to not setup the spark ui and we don't need the kaggle token to download the data, we can just use the kaggle api.

# The minecraft review

In [9]:
!curl -L -o minecraft-blocks-items-mobs-biomes-etc.zip \
  https://www.kaggle.com/api/v1/datasets/download/madelinee/minecraft-blocks-items-mobs-biomes-etc
!unzip minecraft-blocks-items-mobs-biomes-etc.zip -d minecraft-blocks-items-mobs-biomes-etc/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 20959  100 20959    0     0  32806      0 --:--:-- --:--:-- --:--:-- 32806
Archive:  minecraft-blocks-items-mobs-biomes-etc.zip
  inflating: minecraft-blocks-items-mobs-biomes-etc/Biomes.csv  
  inflating: minecraft-blocks-items-mobs-biomes-etc/BlockGeography.csv  
  inflating: minecraft-blocks-items-mobs-biomes-etc/Dimensions.csv  
  inflating: minecraft-blocks-items-mobs-biomes-etc/FaunaGeography.csv  
  inflating: minecraft-blocks-items-mobs-biomes-etc/Flora.csv  
  inflating: minecraft-blocks-items-mobs-biomes-etc/FloraGeography.csv  
  inflating: minecraft-blocks-items-mobs-biomes-etc/Food.csv  
  inflating: minecraft-blocks-items-mobs-biomes-etc/Ingredients.csv  
  inflating: minecraft-blocks-items-mobs-biomes-etc/LandscapeBlocks.csv  
  infla

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum, year, to_date, lit
import plotly.express as px
import pandas as pd

# Load the CSVs with schema inference
# We specifically need Mobs, ManMadeBlocks, and Food for this analysis
mobs_df = spark.read.csv("minecraft-blocks-items-mobs-biomes-etc/Mobs.csv", header=True, inferSchema=True)
blocks_df = spark.read.csv("minecraft-blocks-items-mobs-biomes-etc/ManMadeBlocks.csv", header=True, inferSchema=True)
food_df = spark.read.csv("minecraft-blocks-items-mobs-biomes-etc/Food.csv", header=True, inferSchema=True)

# Clean/Cast Date columns if inference failed (Metadata says sc:Date)
mobs_df = mobs_df.withColumn("debutDate", to_date(col("debutDate")))
blocks_df = blocks_df.withColumn("debutDate", to_date(col("debutDate")))
food_df = food_df.withColumn("debutDate", to_date(col("debutDate")))

In [11]:
# Group by Debut Date (or Year) to see trends over time
mob_evolution = mobs_df.groupBy(year("debutDate").alias("Year")) \
    .agg(
        avg("healthPoints").alias("Avg_Health"),
        avg("maxDamage").alias("Avg_Damage"),
        count("ID").alias("New_Mobs_Added")
    ) \
    .orderBy("Year")

# Convert to Pandas for visualization (Result is small: ~15 rows)
mob_pd = mob_evolution.toPandas()

In [12]:
# Create a dual-axis chart or two line charts
fig = px.line(mob_pd, x="Year", y=["Avg_Health", "Avg_Damage"],
              markers=True,
              title="Minecraft Power Creep: Mob Stats by Introduction Year")
fig.show()

In [13]:
# Calculate complexity ratio: Interactable blocks / Total blocks per year
block_complexity = blocks_df.groupBy(year("debutDate").alias("Year")) \
    .agg(
        sum("interactable").alias("Interactive_Blocks"),
        count("ID").alias("Total_Blocks")
    ) \
    .withColumn("Complexity_Ratio", col("Interactive_Blocks") / col("Total_Blocks")) \
    .orderBy("Year")

block_pd = block_complexity.toPandas()

In [14]:
fig = px.bar(block_pd, x="Year", y="Complexity_Ratio",
             title="The Rise of Machines: % of New Blocks That Are Interactable",
             labels={"Complexity_Ratio": "Interactivity Score (0-1)"})
fig.show()

In [15]:
!curl -L -o minecraft-1-6-1-craftingsmelting-database.zip\
  https://www.kaggle.com/api/v1/datasets/download/sashacooper/minecraft-1-6-1-craftingsmelting-database

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 72141  100 72141    0     0   135k      0 --:--:-- --:--:-- --:--:--  135k


In [16]:
!unzip minecraft-1-6-1-craftingsmelting-database.zip -d minecraft-1-6-1-craftingsmelting-database/

Archive:  minecraft-1-6-1-craftingsmelting-database.zip
  inflating: minecraft-1-6-1-craftingsmelting-database/Minecraft_1.6.1_Database.xlsx  
  inflating: minecraft-1-6-1-craftingsmelting-database/Minecraft_1.6.1_Database_Model.pdf  


In [17]:
%pip install -q openpyxl
import pandas as pd

excel_file_path = "/content/minecraft-1-6-1-craftingsmelting-database/Minecraft_1.6.1_Database.xlsx"
pd_df = pd.read_excel(excel_file_path, sheet_name=None) # Read all sheets into a dictionary of DataFrames

# Convert each sheet's DataFrame to a Spark DataFrame
# The Excel file has multiple sheets, so we'll create a dictionary of Spark DataFrames
spark_excel_data = {}
for sheet_name, df in pd_df.items():
    # Spark doesn't handle all pandas types directly, convert objects to string for compatibility
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str)
    spark_excel_data[sheet_name] = spark.createDataFrame(df)
    print(f"Created Spark DataFrame for sheet: {sheet_name}")

# To access a specific sheet, for example, the first one found:
# first_sheet_name = next(iter(spark_excel_data))
# first_sheet_spark_df = spark_excel_data[first_sheet_name]
# first_sheet_spark_df.show()

Created Spark DataFrame for sheet: Items
Created Spark DataFrame for sheet: Recipes
Created Spark DataFrame for sheet: Recipe Components
Created Spark DataFrame for sheet: Categories
Created Spark DataFrame for sheet: Category Components


In [18]:
for sheet_name, spark_df in spark_excel_data.items():
    print(f"\n--- Schema for sheet: {sheet_name} ---")
    spark_df.printSchema()


--- Schema for sheet: Items ---
root
 |-- Item ID: long (nullable = true)
 |-- In-Game ID: string (nullable = true)
 |-- Name: string (nullable = true)


--- Schema for sheet: Recipes ---
root
 |-- Recipe ID: long (nullable = true)
 |-- Item ID: long (nullable = true)
 |-- Type: string (nullable = true)
 |-- Output Number: long (nullable = true)


--- Schema for sheet: Recipe Components ---
root
 |-- Recipe Component Index: long (nullable = true)
 |-- Recipe ID: long (nullable = true)
 |-- Component: string (nullable = true)
 |-- Category ID: double (nullable = true)
 |-- Location: long (nullable = true)


--- Schema for sheet: Categories ---
root
 |-- Category ID: long (nullable = true)
 |-- Category Name: string (nullable = true)


--- Schema for sheet: Category Components ---
root
 |-- Category Component Index: long (nullable = true)
 |-- Category ID: long (nullable = true)
 |-- Component: string (nullable = true)



We are going Analyze the Minecraft 1.6.1 crafting database from minecraft-1-6-1-craftingsmelting-database/Minecraft_1.6.1_Database.xlsx to determine item crafting complexity, an item is more complex the more step it required to make it, in a second step we are also gonna integrate the complexity of acquiring source items using our other dataset.

## Complexity Pipeline (Implementation)

This section builds a reproducible Spark pipeline with 4 phases:
1) data contracts + normalization,
2) crafting graph construction,
3) source acquisition difficulty,
4) iterative composite complexity scoring.

In [19]:
import time
from contextlib import contextmanager

from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

@contextmanager
def timed(label: str):
    start = time.time()
    yield
    elapsed = time.time() - start
    print(f"[TIMER] {label}: {elapsed:.2f}s")


def profile_df(df, name: str, key_cols=None):
    key_cols = key_cols or []
    row_count = df.count()
    null_exprs = [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]
    nulls = df.select(*null_exprs).collect()[0].asDict() if df.columns else {}
    result = {'name': name, 'rows': row_count, 'nulls': nulls}
    if key_cols:
        duplicate_key_rows = row_count - df.select(*key_cols).dropDuplicates().count()
        result['duplicate_key_rows'] = duplicate_key_rows
    print(f"[PROFILE] {name}: rows={row_count}")
    if key_cols:
        print(f"[PROFILE] {name}: duplicate key rows on {key_cols} = {result['duplicate_key_rows']}")
    return result


def persist_df(df, name: str, level=StorageLevel.MEMORY_AND_DISK):
    persisted = df.persist(level)
    _ = persisted.count()
    print(f"[CACHE] {name} persisted with {level}")
    return persisted


def assert_non_empty(df, name: str):
    if df.rdd.isEmpty():
        raise ValueError(f"{name} is empty")


def assert_columns(df, expected_cols, name: str):
    missing = sorted(set(expected_cols) - set(df.columns))
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")


def canonical_tokens(column_name):
    cleaned = F.regexp_replace(F.lower(F.trim(F.col(column_name))), r'[^a-z0-9 ]', ' ')
    compact = F.regexp_replace(cleaned, r'\s+', ' ')
    return F.array_sort(F.array_distinct(F.filter(F.split(compact, ' '), lambda x: x != '')))


def compact_name(column_name):
    return F.regexp_replace(F.lower(F.trim(F.col(column_name))), r'[^a-z0-9]', '')

In [20]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType
from pyspark.sql.window import Window

def normalize_text(column_name):
    return F.lower(F.trim(F.regexp_replace(F.col(column_name), r'\s+', ' ')))

def first_existing_path(candidates):
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]

crafting_dir = first_existing_path([
    'minecraft-1-6-1-craftingsmelting-database',
    '/content/minecraft-1-6-1-craftingsmelting-database'
])
source_dir = first_existing_path([
    'minecraft-blocks-items-mobs-biomes-etc',
    '/content/minecraft-blocks-items-mobs-biomes-etc'
])

print('Crafting dir:', crafting_dir)
print('Source dir  :', source_dir)

Crafting dir: minecraft-1-6-1-craftingsmelting-database
Source dir  : minecraft-blocks-items-mobs-biomes-etc


In [21]:
required_sheets = {'Items', 'Recipes', 'Recipe Components', 'Categories', 'Category Components'}
available_sheets = set(spark_excel_data.keys())
missing_sheets = required_sheets - available_sheets
if missing_sheets:
    raise ValueError(f'Missing required sheets: {sorted(missing_sheets)}')

df_items_raw = spark_excel_data['Items']
df_recipes_raw = spark_excel_data['Recipes']
df_recipe_comps_raw = spark_excel_data['Recipe Components']
df_categories_raw = spark_excel_data['Categories']
df_cat_comps_raw = spark_excel_data['Category Components']

def canonical_column_name(column_name):
    return ''.join(character.lower() for character in column_name if character.isalnum())

def resolve_column(df, aliases):
    by_canonical = {canonical_column_name(column): column for column in df.columns}
    for alias in aliases:
        key = canonical_column_name(alias)
        if key in by_canonical:
            return by_canonical[key]
    raise ValueError(f'Could not resolve any of {aliases} in columns {df.columns}')

def resolve_optional_column(df, aliases):
    by_canonical = {canonical_column_name(column): column for column in df.columns}
    for alias in aliases:
        key = canonical_column_name(alias)
        if key in by_canonical:
            return by_canonical[key]
    return None

def safe_bigint(column_name):
    return F.expr(f'try_cast(`{column_name}` as BIGINT)')

def safe_double(column_name):
    return F.expr(f'try_cast(`{column_name}` as DOUBLE)')

items_id_col = resolve_column(df_items_raw, ['ID', 'Id', 'Item ID'])
items_name_col = resolve_column(df_items_raw, ['Name', 'Item Name'])
recipes_id_col = resolve_column(df_recipes_raw, ['ID', 'Id', 'Recipe ID'])
recipes_item_id_col = resolve_column(df_recipes_raw, ['Item ID', 'ItemID'])
recipes_output_col = resolve_column(df_recipes_raw, ['Output Number', 'OutputNumber', 'Number'])
recipe_comps_recipe_id_col = resolve_column(df_recipe_comps_raw, ['Recipe ID', 'RecipeID'])
recipe_comps_category_id_col = resolve_column(df_recipe_comps_raw, ['Category ID', 'CategoryID'])
recipe_comps_component_col = resolve_column(df_recipe_comps_raw, ['Component'])
recipe_comps_number_col = resolve_optional_column(df_recipe_comps_raw, ['Number', 'Amount'])
categories_id_col = resolve_column(df_categories_raw, ['ID', 'Id', 'Category ID'])
cat_comps_category_id_col = resolve_column(df_cat_comps_raw, ['Category ID', 'CategoryID'])
cat_comps_component_col = resolve_column(df_cat_comps_raw, ['Component'])

# Validate required semantic columns were found
_ = categories_id_col

df_items = df_items_raw.select(
    safe_bigint(items_id_col).alias('Item_ID'),
    F.col(items_name_col).alias('Item_Name_Raw'),
    normalize_text(items_name_col).alias('Item_Name_Norm')
)

df_recipes = df_recipes_raw.select(
    safe_bigint(recipes_id_col).alias('Recipe_ID'),
    safe_bigint(recipes_item_id_col).alias('Item_ID'),
    F.coalesce(safe_double(recipes_output_col), F.lit(1.0)).alias('Yield')
)

df_recipe_comps = df_recipe_comps_raw.select(
    F.monotonically_increasing_id().alias('Component_Row_ID'),
    safe_bigint(recipe_comps_recipe_id_col).alias('Recipe_ID'),
    safe_bigint(recipe_comps_category_id_col).alias('Category_ID'),
    normalize_text(recipe_comps_component_col).alias('Component_Norm'),
    F.coalesce(safe_double(recipe_comps_number_col), F.lit(1.0)).alias('Input_Qty') if recipe_comps_number_col else F.lit(1.0).alias('Input_Qty')
)

df_cat_comps = df_cat_comps_raw.select(
    safe_bigint(cat_comps_category_id_col).alias('Category_ID'),
    normalize_text(cat_comps_component_col).alias('Category_Component_Norm')
)

assert_columns(df_items, ['Item_ID', 'Item_Name_Raw', 'Item_Name_Norm'], 'df_items')
assert_columns(df_recipes, ['Recipe_ID', 'Item_ID', 'Yield'], 'df_recipes')
assert_columns(df_recipe_comps, ['Component_Row_ID', 'Recipe_ID', 'Category_ID', 'Component_Norm', 'Input_Qty'], 'df_recipe_comps')
assert_columns(df_cat_comps, ['Category_ID', 'Category_Component_Norm'], 'df_cat_comps')

assert_non_empty(df_items, 'df_items')
assert_non_empty(df_recipes, 'df_recipes')
assert_non_empty(df_recipe_comps, 'df_recipe_comps')

profile_df(df_items, 'df_items', key_cols=['Item_ID'])
profile_df(df_recipes, 'df_recipes', key_cols=['Recipe_ID'])
profile_df(df_recipe_comps, 'df_recipe_comps')
profile_df(df_cat_comps, 'df_cat_comps')

print('Contracts validated. Normalized core tables ready.')

[PROFILE] df_items: rows=262
[PROFILE] df_items: duplicate key rows on ['Item_ID'] = 0
[PROFILE] df_recipes: rows=297
[PROFILE] df_recipes: duplicate key rows on ['Recipe_ID'] = 0
[PROFILE] df_recipe_comps: rows=1294
[PROFILE] df_cat_comps: rows=54
Contracts validated. Normalized core tables ready.


In [22]:
# Direct input slots (no category expansion)
direct_slots = df_recipe_comps.filter(F.col('Category_ID').isNull()).select(
    'Recipe_ID', 'Component_Row_ID',
    F.col('Component_Norm').alias('Input_Item_Norm'),
    'Input_Qty'
)

# Category-based slots expanded to candidate components
category_slots = df_recipe_comps.filter(F.col('Category_ID').isNotNull()).join(
    df_cat_comps,
    on='Category_ID',
    how='inner'
).select(
    'Recipe_ID', 'Component_Row_ID',
    F.col('Category_Component_Norm').alias('Input_Item_Norm'),
    'Input_Qty'
)

recipe_input_candidates = direct_slots.unionByName(category_slots).dropna(subset=['Input_Item_Norm'])

# Heuristic augmentation: slot scarcity from number of valid candidates per recipe slot
slot_candidate_counts = recipe_input_candidates.groupBy('Recipe_ID', 'Component_Row_ID').agg(
    F.countDistinct('Input_Item_Norm').alias('Candidate_Count')
).withColumn(
    'Slot_Scarcity',
    F.when(F.col('Candidate_Count') > 0, 1.0 / F.col('Candidate_Count')).otherwise(F.lit(1.0))
)

recipe_outputs = df_recipes.join(df_items.select('Item_ID', 'Item_Name_Norm'), on='Item_ID', how='inner').select(
    'Recipe_ID',
    F.col('Item_Name_Norm').alias('Output_Item_Norm'),
    F.coalesce(F.col('Yield'), F.lit(1.0)).alias('Yield')
)

crafting_graph = recipe_outputs.join(recipe_input_candidates, on='Recipe_ID', how='inner').join(
    slot_candidate_counts,
    on=['Recipe_ID', 'Component_Row_ID'],
    how='left'
).select(
    'Recipe_ID',
    'Output_Item_Norm',
    'Component_Row_ID',
    'Input_Item_Norm',
    'Input_Qty',
    'Yield',
    F.coalesce(F.col('Slot_Scarcity'), F.lit(1.0)).alias('Slot_Scarcity')
)

recipe_slot_requirements = crafting_graph.select('Recipe_ID', 'Component_Row_ID').distinct().groupBy('Recipe_ID').agg(
    F.count('*').alias('Required_Slots')
)

crafting_graph = persist_df(crafting_graph, 'crafting_graph')
recipe_slot_requirements = persist_df(recipe_slot_requirements, 'recipe_slot_requirements')

print('Crafting graph built.')
print('Candidate edges:', crafting_graph.count())
print('Recipes        :', recipe_outputs.select('Recipe_ID').distinct().count())
profile_df(crafting_graph, 'crafting_graph')

[CACHE] crafting_graph persisted with Disk Memory Serialized 1x Replicated
[CACHE] recipe_slot_requirements persisted with Disk Memory Serialized 1x Replicated
Crafting graph built.
Candidate edges: 1756
Recipes        : 297
[PROFILE] crafting_graph: rows=1756


{'name': 'crafting_graph',
 'rows': 1756,
 'nulls': {'Recipe_ID': 0,
  'Output_Item_Norm': 0,
  'Component_Row_ID': 0,
  'Input_Item_Norm': 0,
  'Input_Qty': 0,
  'Yield': 0,
  'Slot_Scarcity': 0}}

In [27]:
# Source dataset loading (acquisition side)
with timed('Load source datasets'):
    df_mobs = spark.read.csv(f'{source_dir}/Mobs.csv', header=True, inferSchema=True)
    df_mob_drops = spark.read.csv(f'{source_dir}/MobIngredientDrops.csv', header=True, inferSchema=True)
    df_flora = spark.read.csv(f'{source_dir}/Flora.csv', header=True, inferSchema=True)
    df_blocks = spark.read.csv(f'{source_dir}/LandscapeBlocks.csv', header=True, inferSchema=True)
    df_ingredients = spark.read.csv(f'{source_dir}/Ingredients.csv', header=True, inferSchema=True)

# Mob difficulty heuristic (health + damage + drop multiplicity pressure)
mob_scores = df_mobs.select(
    F.col('ID').alias('Mob_ID'),
    (
        F.coalesce(F.col('healthPoints').cast(DoubleType()), F.lit(10.0)) +
        F.coalesce(F.col('maxDamage').cast(DoubleType()), F.lit(2.0))
    ).alias('Mob_Score')
)

drop_pressure = df_mob_drops.groupBy('mobID').agg(F.count('*').alias('Mob_Drop_Variants')).withColumn(
    'Drop_Pressure',
    1.0 + (1.0 / F.col('Mob_Drop_Variants'))
)

mob_drop_enriched = df_mob_drops.join(
    mob_scores,
    df_mob_drops['mobID'] == mob_scores['Mob_ID'],
    'inner'
).join(
    drop_pressure,
    on='mobID',
    how='left'
).join(
    df_ingredients.select(F.col('ID').alias('Ingredient_ID'), normalize_text('name').alias('Source_Name_Norm')),
    df_mob_drops['ingredientID'] == F.col('Ingredient_ID'),
    'inner'
).select(
    'Source_Name_Norm',
    'Mob_Score',
    F.coalesce(F.col('Drop_Pressure'), F.lit(1.0)).alias('Drop_Pressure'),
    (F.col('Mob_Score') * F.coalesce(F.col('Drop_Pressure'), F.lit(1.0))).alias('Base_Cost')
)

mob_drop_costs = mob_drop_enriched.select('Source_Name_Norm', 'Base_Cost')
flora_costs = df_flora.select(normalize_text('name').alias('Source_Name_Norm'), F.lit(1.0).alias('Base_Cost'))
block_costs = df_blocks.select(normalize_text('name').alias('Source_Name_Norm'), F.lit(1.0).alias('Base_Cost'))

source_item_costs = mob_drop_costs.unionByName(flora_costs).unionByName(block_costs).groupBy('Source_Name_Norm').agg(
    F.min('Base_Cost').alias('Acquisition_Cost')
)

# Build direct-source prior components used by C0(i)
source_occurrence = df_mob_drops.join(
    df_ingredients.select(F.col('ID').alias('Ingredient_ID'), normalize_text('name').alias('Source_Name_Norm')),
    df_mob_drops['ingredientID'] == F.col('Ingredient_ID'),
    'inner'
).groupBy('Source_Name_Norm').agg(F.count('*').alias('Source_Occurrences'))

source_component_features = source_item_costs.join(source_occurrence, on='Source_Name_Norm', how='left').join(
    mob_drop_enriched.groupBy('Source_Name_Norm').agg(F.max('Mob_Score').alias('Mob_Danger_Raw')),
    on='Source_Name_Norm',
    how='left'
).withColumn(
    'Source_Rarity',
    F.when(F.col('Source_Occurrences').isNull(), F.lit(1.0)).otherwise(1.0 / (1.0 + F.col('Source_Occurrences')))
).withColumn(
    'Mob_Danger',
    F.coalesce(F.col('Mob_Danger_Raw'), F.lit(0.0))
).select(
    'Source_Name_Norm',
    'Acquisition_Cost',
    'Source_Rarity',
    'Mob_Danger'
)

source_component_features = persist_df(source_component_features, 'source_component_features')
source_item_costs = persist_df(source_item_costs, 'source_item_costs')
assert_non_empty(source_item_costs, 'source_item_costs')
profile_df(source_item_costs, 'source_item_costs', key_cols=['Source_Name_Norm'])
print('Source acquisition table ready:', source_item_costs.count(), 'rows')

[TIMER] Load source datasets: 1.21s
[CACHE] source_component_features persisted with Disk Memory Serialized 1x Replicated
[CACHE] source_item_costs persisted with Disk Memory Serialized 1x Replicated
[PROFILE] source_item_costs: rows=266
[PROFILE] source_item_costs: duplicate key rows on ['Source_Name_Norm'] = 0
Source acquisition table ready: 266 rows


In [ ]:
# Canonical name map between crafting item names and source names
# Strategy: exact match + alias normalization + token-based fallback (threshold-gated)
crafting_names = df_items.select(F.col('Item_Name_Norm')).distinct()
source_names = source_item_costs.select(F.col('Source_Name_Norm')).distinct()

# Lightweight alias table for known naming divergences
alias_pairs = [
    ('wood planks', 'oak wood planks'),
    ('planks', 'oak wood planks'),
    ('stone', 'cobblestone'),
    ('smooth stone', 'stone'),
    ('sugar cane', 'sugar canes'),
    ('redstone dust', 'redstone'),
    ('coal ore', 'coal'),
    ('iron ore', 'iron ore block'),
    ('gold ore', 'gold ore block'),
    ('oak log', 'oak wood'),
    ('spruce log', 'spruce wood'),
    ('birch log', 'birch wood'),
    ('jungle log', 'jungle wood'),
    ('acacia log', 'acacia wood'),
    ('dark oak log', 'dark oak wood')
]
alias_df = spark.createDataFrame(alias_pairs, ['Item_Name', 'Alias_Source_Name'])

exact_map = crafting_names.join(
    F.broadcast(source_names),
    crafting_names['Item_Name_Norm'] == source_names['Source_Name_Norm'],
    'inner'
).select(
    F.col('Item_Name_Norm').alias('Item_Name'),
    F.col('Source_Name_Norm').alias('Matched_Source_Name'),
    F.lit('exact').alias('Match_Type'),
    F.lit(1.0).alias('Similarity_Score')
)

alias_map = crafting_names.join(
    F.broadcast(alias_df),
    crafting_names['Item_Name_Norm'] == alias_df['Item_Name'],
    'inner'
).join(
    F.broadcast(source_names),
    alias_df['Alias_Source_Name'] == source_names['Source_Name_Norm'],
    'inner'
).select(
    crafting_names['Item_Name_Norm'].alias('Item_Name'),
    source_names['Source_Name_Norm'].alias('Matched_Source_Name'),
    F.lit('alias').alias('Match_Type'),
    F.lit(0.95).alias('Similarity_Score')
)

# Aggressive normalization stage: compare simplified token signatures
STOP_TOKENS = ['minecraft', 'item', 'block', 'ore', 'wood', 'wooden', 'raw']

def normalized_signature(col_name):
    cleaned = F.regexp_replace(F.lower(F.col(col_name)), r'[^a-z0-9 ]', ' ')
    tokens = F.split(F.regexp_replace(cleaned, r'\s+', ' '), ' ')
    filtered = F.expr(
        f"filter({tokens._jc.toString()}, x -> x <> '' AND NOT array_contains(array({','.join([repr(t) for t in STOP_TOKENS])}), x))"
    )
    return F.array_sort(F.array_distinct(filtered))

crafting_norm = crafting_names.select(
    F.col('Item_Name_Norm').alias('Item_Name'),
    normalized_signature('Item_Name_Norm').alias('Norm_Tokens')
).withColumn('Norm_Key', F.concat_ws(' ', F.col('Norm_Tokens')))

source_norm = source_names.select(
    F.col('Source_Name_Norm').alias('Matched_Source_Name'),
    normalized_signature('Source_Name_Norm').alias('Norm_Tokens')
).withColumn('Norm_Key', F.concat_ws(' ', F.col('Norm_Tokens')))

normalized_exact_map = crafting_norm.join(
    F.broadcast(source_norm),
    on='Norm_Key',
    how='inner'
).filter(F.length(F.col('Norm_Key')) > 0).select(
    'Item_Name',
    'Matched_Source_Name',
    F.lit('normalized_exact').alias('Match_Type'),
    F.lit(0.90).alias('Similarity_Score')
)

matched_so_far = exact_map.select('Item_Name').unionByName(alias_map.select('Item_Name')).unionByName(
    normalized_exact_map.select('Item_Name')
).distinct()

unmatched_after_exact_alias = crafting_names.select(F.col('Item_Name_Norm').alias('Item_Name')).join(
    matched_so_far,
    on='Item_Name',
    how='left_anti'
)

crafting_tokens = unmatched_after_exact_alias.select(
    'Item_Name',
    canonical_tokens('Item_Name').alias('Item_Tokens'),
    compact_name('Item_Name').alias('Item_Compact')
)

source_tokens = source_names.select(
    F.col('Source_Name_Norm').alias('Matched_Source_Name'),
    canonical_tokens('Source_Name_Norm').alias('Source_Tokens'),
    compact_name('Source_Name_Norm').alias('Source_Compact')
)

# Token fallback with combined lexical similarity score and strict acceptance threshold
TOKEN_MATCH_THRESHOLD = 0.45
candidate_pairs = crafting_tokens.crossJoin(F.broadcast(source_tokens)).withColumn(
    'Token_Intersect',
    F.size(F.array_intersect(F.col('Item_Tokens'), F.col('Source_Tokens')))
).withColumn(
    'Token_Union',
    F.size(F.array_union(F.col('Item_Tokens'), F.col('Source_Tokens')))
).withColumn(
    'Item_Token_Count',
    F.size(F.col('Item_Tokens'))
).withColumn(
    'Source_Token_Count',
    F.size(F.col('Source_Tokens'))
).withColumn(
    'Jaccard',
    F.when(F.col('Token_Union') > 0, F.col('Token_Intersect') / F.col('Token_Union')).otherwise(F.lit(0.0))
).withColumn(
    'Dice',
    F.when(
        (F.col('Item_Token_Count') + F.col('Source_Token_Count')) > 0,
        (F.lit(2.0) * F.col('Token_Intersect')) / (F.col('Item_Token_Count') + F.col('Source_Token_Count'))
    ).otherwise(F.lit(0.0))
).withColumn(
    'Overlap',
    F.when(
        F.least(F.col('Item_Token_Count'), F.col('Source_Token_Count')) > 0,
        F.col('Token_Intersect') / F.least(F.col('Item_Token_Count'), F.col('Source_Token_Count'))
    ).otherwise(F.lit(0.0))
).withColumn(
    'Compact_Contains',
    F.when(
        F.col('Source_Compact').contains(F.col('Item_Compact')) | F.col('Item_Compact').contains(F.col('Source_Compact')),
        F.lit(1.0)
    ).otherwise(F.lit(0.0))
).withColumn(
    'Similarity_Score',
    F.col('Jaccard') * F.lit(0.45) + F.col('Dice') * F.lit(0.35) + F.col('Overlap') * F.lit(0.20) + F.col('Compact_Contains') * F.lit(0.10)
)

fallback_ranked = candidate_pairs.withColumn(
    'rank',
    F.row_number().over(Window.partitionBy('Item_Name').orderBy(F.col('Similarity_Score').desc(), F.col('Matched_Source_Name')))
).filter(F.col('rank') == 1)

fallback_map = fallback_ranked.filter(F.col('Similarity_Score') >= F.lit(TOKEN_MATCH_THRESHOLD)).select(
    'Item_Name',
    'Matched_Source_Name',
    F.lit('token_fallback').alias('Match_Type'),
    'Similarity_Score'
)

fallback_rejected = fallback_ranked.filter(F.col('Similarity_Score') < F.lit(TOKEN_MATCH_THRESHOLD)).select(
    'Item_Name',
    'Matched_Source_Name',
    'Similarity_Score'
)

match_candidates = exact_map.unionByName(alias_map).unionByName(normalized_exact_map).unionByName(fallback_map).withColumn(
    'Match_Priority',
    F.when(F.col('Match_Type') == F.lit('exact'), F.lit(1))
     .when(F.col('Match_Type') == F.lit('alias'), F.lit(2))
     .when(F.col('Match_Type') == F.lit('normalized_exact'), F.lit(3))
     .otherwise(F.lit(4))
)

full_map = match_candidates.withColumn(
    'rank',
    F.row_number().over(Window.partitionBy('Item_Name').orderBy(F.col('Match_Priority').asc(), F.col('Similarity_Score').desc(), F.col('Matched_Source_Name').asc()))
).filter(F.col('rank') == 1).select('Item_Name', 'Matched_Source_Name', 'Match_Type', 'Similarity_Score')

item_name_map = crafting_names.select(F.col('Item_Name_Norm').alias('Item_Name')).join(
    full_map,
    on='Item_Name',
    how='left'
).withColumn(
    'Is_Mapped',
    F.col('Matched_Source_Name').isNotNull()
)

mapping_stats = item_name_map.agg(
    F.count('*').alias('Total_Items'),
    F.sum(F.when(F.col('Is_Mapped'), 1).otherwise(0)).alias('Mapped_Items')
).collect()[0]

total_items = mapping_stats['Total_Items']
mapped_items = mapping_stats['Mapped_Items']
coverage = (mapped_items / total_items) if total_items else 0.0
print(f'Mapping coverage: {mapped_items}/{total_items} ({coverage:.2%})')

rejected_count = fallback_rejected.count()
print('Rejected low-confidence token matches:', rejected_count)
item_name_map.groupBy('Match_Type').count().orderBy('count', ascending=False).show(truncate=False)

# Direct acquisition costs only (from mapped source names)
direct_costs = item_name_map.join(
    source_component_features,
    item_name_map['Matched_Source_Name'] == source_component_features['Source_Name_Norm'],
    'left'
).groupBy('Item_Name').agg(
    F.min('Acquisition_Cost').alias('Direct_Cost'),
    F.max('Source_Rarity').alias('Source_Rarity'),
    F.max('Mob_Danger').alias('Mob_Danger')
)

# Graph-derived features for universal priors (depth, ancestor count, branching)
graph_edges = crafting_graph.select('Output_Item_Norm', 'Input_Item_Norm').distinct().collect()
adjacency = {}
reverse_adjacency = {}
for row in graph_edges:
    out_item = row['Output_Item_Norm']
    in_item = row['Input_Item_Norm']
    adjacency.setdefault(out_item, set()).add(in_item)
    reverse_adjacency.setdefault(in_item, set()).add(out_item)

all_item_names = [r['Item_Name_Norm'] for r in crafting_names.collect()]
_depth_cache = {}

def compute_depth(item, path=None):
    if path is None:
        path = set()
    if item in _depth_cache:
        return _depth_cache[item]
    if item in path:
        return 0
    inputs = adjacency.get(item, set())
    if not inputs:
        _depth_cache[item] = 0
        return 0
    new_path = set(path)
    new_path.add(item)
    depth_val = 1 + max(compute_depth(inp, new_path) for inp in inputs)
    _depth_cache[item] = depth_val
    return depth_val

def compute_ancestor_count(item):
    visited = set()
    stack = list(adjacency.get(item, set()))
    while stack:
        current = stack.pop()
        if current in visited:
            continue
        visited.add(current)
        stack.extend(adjacency.get(current, set()))
    return len(visited)

graph_feature_rows = []
for item_name in all_item_names:
    graph_feature_rows.append((
        item_name,
        float(compute_depth(item_name)),
        float(compute_ancestor_count(item_name)),
        float(len(reverse_adjacency.get(item_name, set())))
    ))

graph_item_features = spark.createDataFrame(
    graph_feature_rows,
    ['Item_Name', 'Dependency_Depth', 'Ancestor_Count', 'Branching_Factor']
)

slot_scarcity_by_item = crafting_graph.groupBy('Output_Item_Norm').agg(
    F.avg('Slot_Scarcity').alias('Slot_Scarcity')
).select(F.col('Output_Item_Norm').alias('Item_Name'), 'Slot_Scarcity')

# Name/class prior heuristic
name_class_features = crafting_names.select(
    F.col('Item_Name_Norm').alias('Item_Name')
).withColumn(
    'Name_Class_Prior',
    F.when(F.col('Item_Name').rlike('sword|axe|pickaxe|shovel|hoe|bow|crossbow|trident'), F.lit(1.8))
     .when(F.col('Item_Name').rlike('helmet|chestplate|leggings|boots|shield'), F.lit(1.6))
     .when(F.col('Item_Name').rlike('ore|ingot|gem|diamond|emerald|netherite|quartz|lapis|redstone'), F.lit(1.4))
     .when(F.col('Item_Name').rlike('block|brick|planks|slab|stairs|wall|fence|door|trapdoor'), F.lit(1.1))
     .when(F.col('Item_Name').rlike('food|bread|stew|soup|apple|carrot|potato|beef|chicken|fish|potion'), F.lit(0.9))
     .otherwise(F.lit(1.0))
)

# Universal prior components for all items: C0(i) = w1*source_rarity + w2*mob_danger + w3*slot_scarcity + w4*name/class prior
W_SOURCE_RARITY = 1.2
W_MOB_DANGER = 1.1
W_SLOT_SCARCITY = 1.0
W_NAME_CLASS = 0.9

def with_minmax(df, value_col, out_col):
    stats = df.agg(F.min(value_col).alias('mn'), F.max(value_col).alias('mx')).collect()[0]
    mn = stats['mn']
    mx = stats['mx']
    if mn is None or mx is None or float(mx) == float(mn):
        return df.withColumn(out_col, F.lit(0.0))
    return df.withColumn(out_col, (F.col(value_col) - F.lit(float(mn))) / F.lit(float(mx) - float(mn)))

prior_components = crafting_names.select(F.col('Item_Name_Norm').alias('Item_Name')).distinct().join(
    direct_costs.select('Item_Name', 'Source_Rarity', 'Mob_Danger'),
    on='Item_Name',
    how='left'
).join(
    slot_scarcity_by_item,
    on='Item_Name',
    how='left'
).join(
    graph_item_features,
    on='Item_Name',
    how='left'
).join(
    name_class_features,
    on='Item_Name',
    how='left'
).withColumn('Source_Rarity', F.coalesce(F.col('Source_Rarity'), F.lit(1.0))) \
 .withColumn('Mob_Danger', F.coalesce(F.col('Mob_Danger'), F.lit(0.0))) \
 .withColumn('Slot_Scarcity', F.coalesce(F.col('Slot_Scarcity'), F.lit(0.7))) \
 .withColumn('Dependency_Depth', F.coalesce(F.col('Dependency_Depth'), F.lit(0.0))) \
 .withColumn('Ancestor_Count', F.coalesce(F.col('Ancestor_Count'), F.lit(0.0))) \
 .withColumn('Branching_Factor', F.coalesce(F.col('Branching_Factor'), F.lit(0.0))) \
 .withColumn('Name_Class_Prior', F.coalesce(F.col('Name_Class_Prior'), F.lit(1.0)))

prior_components = with_minmax(prior_components, 'Mob_Danger', 'Mob_Danger_Norm')
prior_components = with_minmax(prior_components, 'Slot_Scarcity', 'Slot_Scarcity_Norm')
prior_components = with_minmax(prior_components, 'Dependency_Depth', 'Dependency_Depth_Norm')
prior_components = with_minmax(prior_components, 'Ancestor_Count', 'Ancestor_Count_Norm')
prior_components = with_minmax(prior_components, 'Branching_Factor', 'Branching_Factor_Norm')

prior_components = prior_components.withColumn(
    'Graph_Fallback',
    F.col('Dependency_Depth_Norm') * F.lit(0.5) +
    F.col('Ancestor_Count_Norm') * F.lit(0.3) +
    F.col('Branching_Factor_Norm') * F.lit(0.2)
).withColumn(
    'C0_Prior_Cost',
    F.col('Source_Rarity') * F.lit(W_SOURCE_RARITY) +
    F.col('Mob_Danger_Norm') * F.lit(W_MOB_DANGER) +
    F.col('Slot_Scarcity_Norm') * F.lit(W_SLOT_SCARCITY) +
    F.col('Name_Class_Prior') * F.lit(W_NAME_CLASS) +
    F.col('Graph_Fallback')
)

universal_prior_costs = prior_components.select(
    'Item_Name',
    F.col('C0_Prior_Cost').alias('Prior_Cost'),
    'Source_Rarity',
    F.col('Mob_Danger_Norm').alias('Mob_Danger'),
    F.col('Slot_Scarcity_Norm').alias('Slot_Scarcity'),
    'Name_Class_Prior',
    'Dependency_Depth',
    'Ancestor_Count',
    'Branching_Factor'
)

source_costs = universal_prior_costs.join(
    direct_costs.select('Item_Name', 'Direct_Cost'),
    on='Item_Name',
    how='left'
).withColumn(
    'Cost',
    F.when(F.col('Direct_Cost').isNull(), F.col('Prior_Cost')).otherwise(F.least(F.col('Direct_Cost'), F.col('Prior_Cost')))
).groupBy('Item_Name').agg(
    F.min('Cost').alias('Cost'),
    F.min('Direct_Cost').alias('Direct_Cost'),
    F.min('Prior_Cost').alias('Prior_Cost')
)

source_costs = persist_df(source_costs, 'source_costs')
assert_non_empty(source_costs, 'source_costs')
profile_df(source_costs, 'source_costs', key_cols=['Item_Name'])
print('Seed source costs (direct + universal prior):', source_costs.count())

Mapping coverage: 68/258 (26.36%)
Rejected low-confidence token matches: 190
+--------------+-----+
|Match_Type    |count|
+--------------+-----+
|NULL          |190  |
|token_fallback|58   |
|exact         |10   |
+--------------+-----+

[CACHE] source_costs persisted with Disk Memory Serialized 1x Replicated
[PROFILE] source_costs: rows=258
[PROFILE] source_costs: duplicate key rows on ['Item_Name'] = 0
Seed source costs (direct + universal prior): 258


In [32]:
CRAFTING_PENALTY = 0.5
SCARCITY_WEIGHT = 0.25
MAX_ITER = 20
CHECKPOINT_EVERY = 5
UNKNOWN_SLOT_BASE = 4.0
UNKNOWN_SLOT_DECAY = 0.85

# Keep direct (acquisition) and prior separate, then combine with craft via min rule
baseline_costs = source_costs.select('Item_Name', 'Direct_Cost', 'Prior_Cost', 'Cost').withColumn(
    'Direct_Or_Prior',
    F.coalesce(F.col('Direct_Cost'), F.col('Prior_Cost'))
)

known_costs = persist_df(
    baseline_costs.select(F.col('Item_Name'), F.col('Cost')).distinct(),
    'known_costs_seed'
)
iteration_metrics = []
item_recipe_min = None

for iteration in range(1, MAX_ITER + 1):
    with timed(f'Iteration {iteration}'):
        old_count = known_costs.count()

        candidate_costs = crafting_graph.alias('cg').join(
            F.broadcast(known_costs.alias('kc')),
            F.col('cg.Input_Item_Norm') == F.col('kc.Item_Name'),
            'left'
        ).select(
            F.col('cg.Recipe_ID'),
            F.col('cg.Output_Item_Norm'),
            F.col('cg.Component_Row_ID'),
            F.col('cg.Input_Qty'),
            F.col('cg.Yield'),
            F.col('cg.Slot_Scarcity'),
            F.col('kc.Cost').alias('Input_Cost')
        )

        unknown_slot_penalty = UNKNOWN_SLOT_BASE * (UNKNOWN_SLOT_DECAY ** (iteration - 1))

        # Soft handling of unknown slots:
        # - if at least one candidate in the slot is known, use the best known candidate
        # - only penalize slots with no known candidate
        slot_costs = candidate_costs.groupBy('Recipe_ID', 'Output_Item_Norm', 'Component_Row_ID', 'Input_Qty', 'Yield', 'Slot_Scarcity').agg(
            F.min('Input_Cost').alias('Known_Slot_Cost'),
            F.max(F.when(F.col('Input_Cost').isNotNull(), F.lit(1)).otherwise(F.lit(0))).alias('Has_Known_Candidate')
        ).withColumn(
            'Known_Contribution',
            F.when(
                F.col('Has_Known_Candidate') == 1,
                F.col('Known_Slot_Cost') * F.col('Input_Qty') * (1 + F.lit(SCARCITY_WEIGHT) * F.col('Slot_Scarcity'))
            ).otherwise(F.lit(0.0))
        ).withColumn(
            'Is_Unknown_Slot',
            F.when(F.col('Has_Known_Candidate') == 1, F.lit(0)).otherwise(F.lit(1))
        )

        recipe_costs = slot_costs.groupBy('Recipe_ID', 'Output_Item_Norm', 'Yield').agg(
            F.sum('Known_Contribution').alias('Known_Input_Cost'),
            F.sum('Is_Unknown_Slot').alias('Unknown_Slots')
        ).withColumn(
            'Recipe_Cost',
            (
                F.col('Known_Input_Cost') +
                (F.col('Unknown_Slots') * F.lit(unknown_slot_penalty)) +
                F.lit(CRAFTING_PENALTY)
            ) / F.coalesce(F.col('Yield'), F.lit(1.0))
        )

        # Player chooses the cheapest available recipe for this item
        item_recipe_min = recipe_costs.groupBy('Output_Item_Norm').agg(
            F.min('Recipe_Cost').alias('Craft_Cost'),
            F.min('Unknown_Slots').alias('Best_Unknown_Slots')
        ).select(
            F.col('Output_Item_Norm').alias('Item_Name'),
            'Craft_Cost',
            'Best_Unknown_Slots'
        )

        # Realistic obtain rule: C(i) = min(C_direct(i), C_craft(i)),
        # where C_direct falls back to universal prior when direct acquisition is missing.
        updated_costs = baseline_costs.alias('base').join(
            item_recipe_min.alias('craft'),
            on='Item_Name',
            how='left'
        ).join(
            known_costs.alias('old'),
            on='Item_Name',
            how='left'
        ).select(
            F.col('Item_Name'),
            F.col('base.Direct_Cost'),
            F.col('base.Prior_Cost'),
            F.col('base.Direct_Or_Prior').alias('Direct_Baseline_Cost'),
            F.col('craft.Craft_Cost'),
            F.col('craft.Best_Unknown_Slots'),
            F.col('old.Cost').alias('Old_Cost'),
            F.when(
                F.col('craft.Craft_Cost').isNull(),
                F.col('base.Direct_Or_Prior')
            ).otherwise(
                F.least(F.col('base.Direct_Or_Prior'), F.col('craft.Craft_Cost'))
            ).alias('Route_Cost')
        ).withColumn(
            'Cost',
            F.when(F.col('Old_Cost').isNull(), F.col('Route_Cost')).otherwise(F.least(F.col('Old_Cost'), F.col('Route_Cost')))
        ).drop('Old_Cost', 'Route_Cost').dropna(subset=['Item_Name', 'Cost'])

        new_count = updated_costs.select('Item_Name').distinct().count()
        added_count = new_count - old_count

        known_costs = updated_costs.select('Item_Name', 'Cost').distinct()

        if iteration % CHECKPOINT_EVERY == 0:
            known_costs = known_costs.checkpoint(eager=True)

        iteration_metrics.append((iteration, old_count, new_count, added_count, float(unknown_slot_penalty)))
        print(f'Iteration {iteration}: +{added_count} items ({old_count} -> {new_count}), unknown_penalty={unknown_slot_penalty:.4f}')

        if added_count <= 0:
            print(f'Iteration {iteration}: convergence reached at {new_count} known items.')
            break

known_costs = persist_df(known_costs, 'known_costs_final')

if item_recipe_min is None:
    item_recipe_min = spark.createDataFrame([], 'Item_Name string, Craft_Cost double, Best_Unknown_Slots bigint')

# Build final table with direct/craft decomposition and explicit min rule outcome
final_complexity_df = baseline_costs.alias('base').join(
    known_costs.alias('k'),
    on='Item_Name',
    how='left'
).join(
    item_recipe_min.alias('craft'),
    on='Item_Name',
    how='left'
).select(
    F.col('Item_Name'),
    F.col('base.Direct_Cost'),
    F.col('base.Prior_Cost'),
    F.col('craft.Craft_Cost'),
    F.coalesce(F.col('k.Cost'), F.col('base.Direct_Or_Prior')).alias('Cost')
).orderBy(F.col('Cost').desc())

iteration_metrics_df = spark.createDataFrame(
    iteration_metrics,
    schema=['Iteration', 'Known_Before', 'Known_After', 'Added', 'Unknown_Penalty']
)

print('Final known items:', final_complexity_df.count())
iteration_metrics_df.show(truncate=False)

[CACHE] known_costs_seed persisted with Disk Memory Serialized 1x Replicated
Iteration 1: +0 items (258 -> 258), unknown_penalty=4.0000
Iteration 1: convergence reached at 258 known items.
[TIMER] Iteration 1: 11.93s
[CACHE] known_costs_final persisted with Disk Memory Serialized 1x Replicated
Final known items: 258
+---------+------------+-----------+-----+---------------+
|Iteration|Known_Before|Known_After|Added|Unknown_Penalty|
+---------+------------+-----------+-----+---------------+
|1        |258         |258        |0    |4.0            |
+---------+------------+-----------+-----+---------------+



In [33]:
# Diagnostics and explainability outputs
all_crafting_outputs = crafting_graph.select(F.col('Output_Item_Norm').alias('Item_Name')).distinct()
unresolved_items = all_crafting_outputs.join(final_complexity_df.select('Item_Name'), on='Item_Name', how='left_anti')

print('Top 20 Most Complex Items:')
final_complexity_df.show(20, truncate=False)

non_null_count = final_complexity_df.filter(F.col('Cost').isNotNull()).count()
total_outputs = all_crafting_outputs.count()
non_null_coverage = (non_null_count / total_outputs) if total_outputs else 0.0
print(f'Final non-null effort coverage: {non_null_count}/{total_outputs} ({non_null_coverage:.2%})')

print('Unresolved outputs count:', unresolved_items.count())
unresolved_items.orderBy('Item_Name').show(20, truncate=False)

print('Token fallback rejections:', fallback_rejected.count())
print('Match type distribution:')
item_name_map.groupBy('Match_Type').count().orderBy(F.col('count').desc()).show(truncate=False)

print('Unknown penalty evolution by iteration:')
iteration_metrics_df.select('Iteration', 'Unknown_Penalty', 'Known_After', 'Added').orderBy('Iteration').show(truncate=False)

# One-hop explainability table for a target item
target_item = 'enchantment table'
target_recipes = crafting_graph.filter(F.col('Output_Item_Norm') == F.lit(target_item)).select('Recipe_ID').distinct()
explain_table = crafting_graph.join(target_recipes, on='Recipe_ID', how='inner').join(
    final_complexity_df.select(F.col('Item_Name').alias('Input_Item_Norm'), F.col('Cost').alias('Input_Cost')),
    on='Input_Item_Norm',
    how='left'
).select('Recipe_ID', 'Output_Item_Norm', 'Component_Row_ID', 'Input_Item_Norm', 'Input_Qty', 'Input_Cost').orderBy('Recipe_ID', 'Component_Row_ID')

print(f'Explainability trace for: {target_item}')
explain_table.show(100, truncate=False)

Top 20 Most Complex Items:
+-----------------+-----------+------------------+------------------+------------------+
|Item_Name        |Direct_Cost|Prior_Cost        |Craft_Cost        |Cost              |
+-----------------+-----------+------------------+------------------+------------------+
|golden hoe       |NULL       |4.148260869565218 |14.640095990589273|4.148260869565218 |
|golden shovel    |NULL       |4.148260869565218 |10.477352283724192|4.148260869565218 |
|golden sword     |NULL       |4.148260869565218 |11.732791702159716|4.148260869565218 |
|golden pickaxe   |NULL       |4.148260869565218 |18.802839697454353|4.148260869565218 |
|golden axe       |NULL       |4.148260869565218 |18.802839697454353|4.148260869565218 |
|iron pickaxe     |NULL       |4.141739130434783 |20.339131376434935|4.141739130434783 |
|iron axe         |NULL       |4.141739130434783 |20.339131376434935|4.141739130434783 |
|iron shovel      |NULL       |4.141739130434783 |10.989449510051053|4.141739130434